# Preparação dos dados de área ardida

* Organizar, filtrar e transformar esta informação para que possa ser usada no cálculo da susceptibilidade, da probabilidade e na validação dos resultados.

In [1]:
import os
import shutil
from glob import glob

import geopandas as gpd
import numpy as np
import rasterio as rio

#from glass.gp.ovl.clipp import clip
from glass.rst.stats import count_region_in_shape

In [2]:
raw_shps = sorted(glob("/code/data/raw/area_ardida/icnf/*/*.shp")) 

#aoi = "/code/data/processed/pnse/aoi/pnse.shp" #alterar consoante a área de estudo
aoi = "/code/data/processed/centro/aoi/centro.shp" #região centro
#ref = "/code/data/processed/pnse/topo/derived/dem_pnse.tif" 
ref = "/code/data/raw/dem_centro/dem_ctr.tif"  #tif da região centro

out_year = "/code/data/processed/centro/area_ardida/yearly_vector" 
out_aoi = "/code/data/processed/centro/area_ardida/yearly_aoi"
out_train = "/code/data/processed/centro/area_ardida/train"
out_valid = "/code/data/processed/centro/area_ardida/valid"

out_rst_count_dir = "/code/data/processed/centro/area_ardida/raster_count"
out_rst_bin_dir = "/code/data/processed/centro/area_ardida/raster_binary"
out_tmp_periods = "/code/data/scratch/tmp_periods/centro/area_ardida"

out_rst = os.path.join(out_rst_count_dir, "rst_ba_1975_2024.tif")
validation_year = 2025

for p in [
    out_year, out_aoi, out_train, out_valid,
    out_rst_count_dir, out_rst_bin_dir, out_tmp_periods
]:
    os.makedirs(p, exist_ok=True)

print("Blocos encontrados:")
for f in raw_shps:
    print("-", f)

Blocos encontrados:
- /code/data/raw/area_ardida/icnf/ardida_1975_1989/ardida_1975_1989.shp
- /code/data/raw/area_ardida/icnf/ardida_1990_1999/ardida_1990_1999.shp
- /code/data/raw/area_ardida/icnf/ardida_2000_2008/ardida_2000_2008.shp
- /code/data/raw/area_ardida/icnf/ardida_2009/ardida_2009.shp
- /code/data/raw/area_ardida/icnf/ardida_2010/ardida_2010.shp
- /code/data/raw/area_ardida/icnf/ardida_2011/ardida_2011.shp
- /code/data/raw/area_ardida/icnf/ardida_2012/ardida_2012.shp
- /code/data/raw/area_ardida/icnf/ardida_2013/ardida_2013.shp
- /code/data/raw/area_ardida/icnf/ardida_2014/ardida_2014.shp
- /code/data/raw/area_ardida/icnf/ardida_2015/ardida_2015.shp
- /code/data/raw/area_ardida/icnf/ardida_2016/ardida_2016.shp
- /code/data/raw/area_ardida/icnf/ardida_2017/ardida_2017.shp
- /code/data/raw/area_ardida/icnf/ardida_2018/ardida_2018.shp
- /code/data/raw/area_ardida/icnf/ardida_2019/ardida_2019.shp
- /code/data/raw/area_ardida/icnf/ardida_2020/ardida_2020.shp
- /code/data/raw/are

In [3]:
gdf_test = gpd.read_file(raw_shps[0])

print(gdf_test.shape)
print(gdf_test.crs)
print(gdf_test.columns.tolist())
print(sorted(gdf_test["Ano"].dropna().unique())[:10])
print(gdf_test["Ano"].value_counts(dropna=False).sort_index().head())

(12654, 3)
EPSG:3763
['Ano', 'AreaHaSIG', 'geometry']
[1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984]
Ano
1975    280
1976    190
1977    100
1978    529
1979    141
Name: count, dtype: int64


In [4]:
#DEGUB
for shp in raw_shps:
    gdf = gpd.read_file(shp)

    print(f"\nBloco: {os.path.basename(shp)}")
    print(f"Total de feições: {len(gdf)}")
    print(f"Geometrias inválidas no bloco: {(~gdf.is_valid).sum()}")

    if "Ano" in gdf.columns:
        anos = sorted(gdf["Ano"].dropna().unique())

        for ano in anos:
            sub = gdf[gdf["Ano"] == ano].copy()
            n_invalid = (~sub.is_valid).sum()

            if n_invalid > 0:
                print(f"  Ano {int(ano)} -> geometrias inválidas: {n_invalid}")


Bloco: ardida_1975_1989.shp
Total de feições: 12654
Geometrias inválidas no bloco: 179
  Ano 1975 -> geometrias inválidas: 1
  Ano 1976 -> geometrias inválidas: 1
  Ano 1978 -> geometrias inválidas: 15
  Ano 1981 -> geometrias inválidas: 1
  Ano 1984 -> geometrias inválidas: 19
  Ano 1985 -> geometrias inválidas: 80
  Ano 1986 -> geometrias inválidas: 14
  Ano 1987 -> geometrias inválidas: 14
  Ano 1988 -> geometrias inválidas: 9
  Ano 1989 -> geometrias inválidas: 25

Bloco: ardida_1990_1999.shp
Total de feições: 10564
Geometrias inválidas no bloco: 431
  Ano 1990 -> geometrias inválidas: 93
  Ano 1991 -> geometrias inválidas: 28
  Ano 1993 -> geometrias inválidas: 30
  Ano 1994 -> geometrias inválidas: 64
  Ano 1995 -> geometrias inválidas: 62
  Ano 1996 -> geometrias inválidas: 124
  Ano 1997 -> geometrias inválidas: 30

Bloco: ardida_2000_2008.shp
Total de feições: 10981
Geometrias inválidas no bloco: 201
  Ano 2001 -> geometrias inválidas: 1
  Ano 2002 -> geometrias inválidas: 6


## Filtragem por área mínima

Aplica-se um filtro por área mínima aos perímetros de incêndio, de modo a manter maior consistência temporal no inventário de áreas ardidas.

In [5]:
# Aplica-se um filtro por área mínima aos perímetros de incêndio
def filtrar_area_minima(gdf, campo_ano="Ano", campo_area_ha="AreaHaSIG"):
    gdf = gdf.copy()
    gdf = gdf[
        ((gdf[campo_ano] <= 1983) & (gdf[campo_area_ha] > 30)) |
        ((gdf[campo_ano] > 1983) & (gdf[campo_area_ha] > 5))
    ]
    return gdf

## Separação dos blocos de área ardida por ano
* dados do ICNF têm as áreas ardidas agregadas por blocos desde 1975 até 2008, sendo necessária a sua separação

In [6]:
#exportar os ficheiros anuais separados
for shp in raw_shps:
    gdf = gpd.read_file(shp)
    n0 = len(gdf)

    gdf = filtrar_area_minima(gdf, campo_ano="Ano", campo_area_ha="AreaHaSIG")
    n1 = len(gdf)

    keep_cols = ["Ano", "AreaHaSIG", "geometry"]
    keep_cols = [c for c in keep_cols if c in gdf.columns]
    gdf = gdf[keep_cols].copy()

    anos = sorted(gdf["Ano"].dropna().unique())

    print(f"\nBloco: {os.path.basename(shp)}")
    print(f"Feições antes do filtro: {n0}")
    print(f"Feições depois do filtro: {n1}")
    print("Anos após filtro:", anos)

    for ano in anos:
        ay = gdf[gdf["Ano"] == ano].copy()
        of = os.path.join(out_year, f"aa_{int(ano)}.shp")
        ay.to_file(of)
        print("gravado:", of, "| n =", len(ay))


Bloco: ardida_1975_1989.shp
Feições antes do filtro: 12654
Feições depois do filtro: 12654
Anos após filtro: [1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989]
gravado: /code/data/processed/centro/area_ardida/yearly_vector/aa_1975.shp | n = 280
gravado: /code/data/processed/centro/area_ardida/yearly_vector/aa_1976.shp | n = 190
gravado: /code/data/processed/centro/area_ardida/yearly_vector/aa_1977.shp | n = 100
gravado: /code/data/processed/centro/area_ardida/yearly_vector/aa_1978.shp | n = 529
gravado: /code/data/processed/centro/area_ardida/yearly_vector/aa_1979.shp | n = 141
gravado: /code/data/processed/centro/area_ardida/yearly_vector/aa_1980.shp | n = 258
gravado: /code/data/processed/centro/area_ardida/yearly_vector/aa_1981.shp | n = 399
gravado: /code/data/processed/centro/area_ardida/yearly_vector/aa_1982.shp | n = 151
gravado: /code/data/processed/centro/area_ardida/yearly_vector/aa_1983.shp | n = 173
gravado: /code/data/processed/cent

## Recorte das áreas ardidas anuais à AOI

In [7]:
from glob import glob
import os

import geopandas as gpd
from shapely import make_valid

year_shps = sorted(glob(os.path.join(out_year, "*.shp")))

aoi_gdf = gpd.read_file(aoi).dissolve()
aoi_boundary = aoi_gdf.geometry.iloc[0].boundary

# área de uma célula do raster de referência
with rio.open(ref) as src:
    res_x = abs(src.transform.a)
    res_y = abs(src.transform.e)

cell_area = res_x * res_y
area_min_global = cell_area          # 1 célula
area_min_limite = 5 * cell_area      # 5 células no limite da AOI

print("Ficheiros anuais:", len(year_shps))
print(f"Resolução raster: {res_x} x {res_y} m")
print(f"Área mínima global: {area_min_global} m²")
print(f"Área mínima no limite da AOI: {area_min_limite} m²")


def apagar_shapefile(path_shp):
    base, _ = os.path.splitext(path_shp)
    for ext in [".shp", ".shx", ".dbf", ".prj", ".cpg", ".qix"]:
        f = base + ext
        if os.path.exists(f):
            os.remove(f)


def corrigir_para_poligonos(geom):
    if geom is None or geom.is_empty:
        return None

    geom = make_valid(geom)

    if geom is None or geom.is_empty:
        return None

    if geom.geom_type in ["Polygon", "MultiPolygon"]:
        return geom

    if geom.geom_type == "GeometryCollection":
        polys = [g for g in geom.geoms if g.geom_type in ["Polygon", "MultiPolygon"] and not g.is_empty]
        if not polys:
            return None
        return polys[0] if len(polys) == 1 else gpd.GeoSeries(polys).union_all()

    return None


def limpar_pre_clip(gdf):
    gdf = gdf.copy()

    gdf["geometry"] = gdf["geometry"].apply(corrigir_para_poligonos)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf = gdf[gdf.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

    return gdf


def limpar_pos_clip(gdf, aoi_boundary, area_min_global, area_min_limite):
    gdf = gdf.copy()

    gdf["geometry"] = gdf["geometry"].apply(corrigir_para_poligonos)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf = gdf[gdf.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

    if gdf.empty:
        return gdf

    # transformar multipartes em partes simples
    gdf = gdf.explode(index_parts=False).reset_index(drop=True)

    gdf["area_m2"] = gdf.geometry.area
    gdf["touches_aoi_boundary"] = gdf.geometry.apply(
        lambda geom: geom.boundary.intersects(aoi_boundary)
    )

    # regra técnica:
    # - remover sempre partes menores ou iguais a 1 célula;
    # - remover no limite da AOI partes até 5 células
    gdf = gdf[
        (gdf["area_m2"] > area_min_global) &
        ~(
            gdf["touches_aoi_boundary"] &
            (gdf["area_m2"] <= area_min_limite)
        )
    ].copy()

    gdf = gdf.drop(columns=["area_m2", "touches_aoi_boundary"])

    return gdf

Ficheiros anuais: 51
Resolução raster: 10.0 x 10.0 m
Área mínima global: 100.0 m²
Área mínima no limite da AOI: 500.0 m²


In [8]:
for shp in year_shps:
    of = os.path.join(out_aoi, os.path.basename(shp))

    gdf = gpd.read_file(shp)

    if gdf.crs != aoi_gdf.crs:
        gdf = gdf.to_crs(aoi_gdf.crs)

    # apenas correcção geométrica antes do clip
    gdf = limpar_pre_clip(gdf)

    # clip exacto
    gdf_clip = gpd.clip(gdf, aoi_gdf)

    # limpeza técnica apenas depois do clip
    gdf_clip = limpar_pos_clip(
        gdf_clip,
        aoi_boundary=aoi_boundary,
        area_min_global=area_min_global,
        area_min_limite=area_min_limite
    )

    apagar_shapefile(of)
    gdf_clip.to_file(of)

    print("clip:", os.path.basename(of), "| n =", len(gdf_clip))

clip: aa_1975.shp | n = 131
clip: aa_1976.shp | n = 93
clip: aa_1977.shp | n = 38
clip: aa_1978.shp | n = 222
clip: aa_1979.shp | n = 76
clip: aa_1980.shp | n = 111
clip: aa_1981.shp | n = 151
clip: aa_1982.shp | n = 63
clip: aa_1983.shp | n = 117
clip: aa_1984.shp | n = 459
clip: aa_1985.shp | n = 698
clip: aa_1986.shp | n = 442
clip: aa_1987.shp | n = 405
clip: aa_1988.shp | n = 177
clip: aa_1989.shp | n = 607
clip: aa_1990.shp | n = 336
clip: aa_1991.shp | n = 339
clip: aa_1992.shp | n = 87
clip: aa_1993.shp | n = 59
clip: aa_1994.shp | n = 304
clip: aa_1995.shp | n = 615
clip: aa_1996.shp | n = 378
clip: aa_1997.shp | n = 291
clip: aa_1998.shp | n = 407
clip: aa_1999.shp | n = 351
clip: aa_2000.shp | n = 543
clip: aa_2001.shp | n = 485
clip: aa_2002.shp | n = 397
clip: aa_2003.shp | n = 348
clip: aa_2004.shp | n = 157
clip: aa_2005.shp | n = 422
clip: aa_2006.shp | n = 189
clip: aa_2007.shp | n = 142
clip: aa_2008.shp | n = 100
clip: aa_2009.shp | n = 178
clip: aa_2010.shp | n = 29

In [9]:
"""year_shps = sorted(glob(os.path.join(out_year, "*.shp")))

print("Ficheiros anuais:", len(year_shps))

for shp in year_shps:
    of = os.path.join(out_aoi, os.path.basename(shp))

    clip(
        inFeat=shp,
        clipFeat=aoi,
        outFeat=of,
        api_gis="ogr2ogr"
    )

    gdf_clip = gpd.read_file(of)
    print("clip:", os.path.basename(of), "| n =", len(gdf_clip))"""

'year_shps = sorted(glob(os.path.join(out_year, "*.shp")))\n\nprint("Ficheiros anuais:", len(year_shps))\n\nfor shp in year_shps:\n    of = os.path.join(out_aoi, os.path.basename(shp))\n\n    clip(\n        inFeat=shp,\n        clipFeat=aoi,\n        outFeat=of,\n        api_gis="ogr2ogr"\n    )\n\n    gdf_clip = gpd.read_file(of)\n    print("clip:", os.path.basename(of), "| n =", len(gdf_clip))'

## Separação Treino/Validação

In [10]:
aoi_shps = sorted(glob(os.path.join(out_aoi, "*.shp")))

for shp in aoi_shps:
    ano = int(os.path.basename(shp).replace("aa_", "").replace(".shp", ""))

    gdf = gpd.read_file(shp)

    if ano == validation_year:
        dst = os.path.join(out_valid, os.path.basename(shp))
        gdf.to_file(dst)
    else:
        dst = os.path.join(out_train, os.path.basename(shp))
        gdf.to_file(dst)

print("Ficheiros de treino:", len(glob(os.path.join(out_train, "*.shp"))))
print("Ficheiros de validação:", len(glob(os.path.join(out_valid, "*.shp"))))

Ficheiros de treino: 50
Ficheiros de validação: 1


## Criação dos rasters acumulados

In [11]:
# Converte células sem ocorrência para nodata no raster final
def zeros_para_nodata(path, nodata=-1):
    with rio.open(path) as src:
        arr = src.read(1)
        profile = src.profile.copy()

    arr = arr.astype("int16")
    arr[arr == 0] = nodata

    profile.update(dtype="int16", nodata=nodata)

    with rio.open(path, "w", **profile) as dst:
        dst.write(arr, 1)

In [12]:
# Agrega os ficheiros anuais de um dado período e gera um raster de contagem por célula
def criar_raster_ba_periodo(
    anos,
    pasta_origem,
    pasta_tmp_base,
    ref,
    out_raster
):
    nome_periodo = f"{min(anos)}_{max(anos)}"
    pasta_tmp = os.path.join(pasta_tmp_base, nome_periodo)

    if os.path.exists(pasta_tmp):
        shutil.rmtree(pasta_tmp)
    os.makedirs(pasta_tmp, exist_ok=True)

    for ano in anos:
        shp = os.path.join(pasta_origem, f"aa_{ano}.shp")
        if os.path.exists(shp):
            for f in glob(shp.replace(".shp", ".*")):
                shutil.copy(f, pasta_tmp)

    count_region_in_shape(
        folder=pasta_tmp,
        ref=ref,
        out=out_raster,
        returnprob=None
    )

    zeros_para_nodata(out_raster)

    print("Raster criado:", out_raster)

In [14]:
from glass.rst.stats import count_region_in_shape
# raster acumulado geral
criar_raster_ba_periodo(
    anos=list(range(1975, 2025)),
    pasta_origem=out_train,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_1975_2024.tif")
)

# raster acumulado compatível com LULC
criar_raster_ba_periodo(
    anos=list(range(1995, 2025)),
    pasta_origem=out_train,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_1995_2024.tif")
)

# janelas temporais 
criar_raster_ba_periodo(
    anos=list(range(1995, 2007)),
    pasta_origem=out_aoi,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_1995_2006.tif")
)

criar_raster_ba_periodo(
    anos=list(range(2007, 2010)),
    pasta_origem=out_aoi,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_2007_2009.tif")
)

criar_raster_ba_periodo(
    anos=list(range(2010, 2015)),
    pasta_origem=out_aoi,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_2010_2014.tif")
)

criar_raster_ba_periodo(
    anos=list(range(2015, 2018)),
    pasta_origem=out_aoi,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_2015_2017.tif")
)


criar_raster_ba_periodo(
    anos=list(range(2018, 2025)),
    pasta_origem=out_aoi,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_2018_2024.tif")
)

         Unable to calculate a centroid for 1 areas


Raster criado: /code/data/processed/centro/area_ardida/raster_count/rst_ba_1975_2024.tif


         Unable to calculate a centroid for 1 areas


Raster criado: /code/data/processed/centro/area_ardida/raster_count/rst_ba_1995_2024.tif
Raster criado: /code/data/processed/centro/area_ardida/raster_count/rst_ba_1995_2006.tif
Raster criado: /code/data/processed/centro/area_ardida/raster_count/rst_ba_2007_2009.tif
Raster criado: /code/data/processed/centro/area_ardida/raster_count/rst_ba_2010_2014.tif


         Unable to calculate a centroid for 1 areas


Raster criado: /code/data/processed/centro/area_ardida/raster_count/rst_ba_2015_2017.tif
Raster criado: /code/data/processed/centro/area_ardida/raster_count/rst_ba_2018_2024.tif


In [15]:
print("Anuais separados:", len(glob(os.path.join(out_year, "*.shp"))))
print("Anuais recortados à AOI:", len(glob(os.path.join(out_aoi, "*.shp"))))
print("Anuais de treino:", len(glob(os.path.join(out_train, "*.shp"))))
print("Anuais de validação:", len(glob(os.path.join(out_valid, "*.shp"))))

Anuais separados: 51
Anuais recortados à AOI: 51
Anuais de treino: 50
Anuais de validação: 1


## Criação dos rasters binários

In [16]:
def contagem_para_binario(src_path, dst_path, nodata=-1):
    with rio.open(src_path) as src:
        arr = src.read(1)
        profile = src.profile.copy()
        src_nodata = src.nodata

    out = np.full(arr.shape, nodata, dtype="int16")

    if src_nodata is None:
        mask = arr > 0
    else:
        mask = arr != src_nodata #corrigir!!!

    out[mask] = 1

    profile.update(dtype="int16", nodata=nodata)

    with rio.open(dst_path, "w", **profile) as dst:
        dst.write(out, 1)

    print("Raster binário criado:", dst_path)

In [17]:
criar_raster_ba_periodo(
    anos=[2025],
    pasta_origem=out_valid,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_2025.tif")
)

Raster criado: /code/data/processed/centro/area_ardida/raster_count/rst_ba_2025.tif


In [18]:
contagem_para_binario(
    os.path.join(out_rst_count_dir, "rst_ba_1995_2024.tif"),
    os.path.join(out_rst_bin_dir, "rst_ba_1995_2024_bin.tif")
)

contagem_para_binario(
    os.path.join(out_rst_count_dir, "rst_ba_2025.tif"),
    os.path.join(out_rst_bin_dir, "rst_ba_2025_bin.tif")
)

Raster binário criado: /code/data/processed/centro/area_ardida/raster_binary/rst_ba_1995_2024_bin.tif
Raster binário criado: /code/data/processed/centro/area_ardida/raster_binary/rst_ba_2025_bin.tif
